# A micrograd neuron, in SKaiNET

Port of the two-input single-neuron example from Andrej Karpathy's [micrograd](https://github.com/karpathy/micrograd) — the canonical "see the autodiff graph" demo. All credit for the original tutorial and graph-visualisation style belongs to Karpathy; this notebook is just a tribute showing the same graph shape coming out of SKaiNET's DAG DSL through our notebook-side `asDot()` renderer.

Original snippet:

```python
from micrograd import nn
n = nn.Neuron(2)
x = [Value(1.0), Value(-2.0)]
y = n(x)
dot = draw_dot(y)
```

**Prerequisite:** run `./gradlew :kotlin-notebook:publishToMavenLocal` from the repo root, then restart the kernel.

In [1]:
USE {
    repositories {
        mavenLocal()
    }
    dependencies {
        implementation("sk.ainet.app:kotlin-notebook:0.25.0")
    }
}

SKaiNET Kotlin Notebook v0.25.0 ready

⚠ SKaiNET SIMD path is NOT active — falling back to scalar CPU kernels. 
 jdk.incubator.vector module not loaded — start the kernel with --add-modules jdk.incubator.vector 
 IntelliJ Kotlin Notebook: Settings → Languages & Frameworks → Kotlin → Kotlin Notebook → JVM options, add --add-modules jdk.incubator.vector . 
 Run checkSimd() for the full diagnostic.

## The translation

micrograd's `Neuron(2)` is `tanh(w · x + b)` over two inputs, with `w` random and `b` zero. SKaiNET 0.25.0 doesn't ship `tanh` as a DAG-DSL primitive, so the notebook integration provides a polyfill that composes the exact identity `tanh(x) = 2*sigmoid(2x) - 1` (see [`docs/upstream/tanh-activation.md`](../../docs/upstream/tanh-activation.md) for the upstream proposal that would replace it with a real primitive). The rendered graph shows the `mulScalar -> sigmoid -> mulScalar -> subScalar` decomposition rather than a single `tanh` block — faithful to what the kernel is actually computing today.

The leaf `Value(1.0)`/`Value(-2.0)` inputs become a `constant` tensor of shape `(1, 2)` so the values show up directly on the graph, the same way Karpathy's `draw_dot` surfaces the leaves. Returning the program from a cell renders the graph as inline SVG via the bundled Graphviz wasm — same pipeline as `draw_dot`, just running on the JVM kernel instead of a Python process.

In [ ]:
import sk.ainet.lang.types.FP32

// n = nn.Neuron(2);  x = [Value(1.0), Value(-2.0)];  y = n(x)
val program = dag {
    val x = constant<FP32, Float>("x") {
        fromArray(floatArrayOf(1.0f, -2.0f), shape = listOf(1, 2))
    }
    val w = parameter<FP32, Float>("w") { shape(2, 1) { ones() } }
    val b = parameter<FP32, Float>("b") { shape(1) { zeros() } }

    val act = add(matmul(x, w), b)
    val y = tanh(act)                 // polyfilled as 2*sigmoid(2x)-1 (see docs/upstream/tanh-activation.md)

    output(y)
}

program.asDot()